# 04 EDA Review: Housing, Income, and Commute

This notebook is the final EDA review notebook for the data foundation phase. It validates the processed files, records the project caveats, and recommends the final analytical structure and chart list.

This is still diagnostic work. Do not treat the charts in this notebook as final polished visuals.

## A. Data Files Loaded

The review uses the processed and interim outputs from the data foundation phase. The app should later read processed CSVs rather than rerunning ingestion.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = ROOT / "reports"
CHART_DIR = REPORTS_DIR / "eda_review_charts"

files = {
    "data_inventory": INTERIM_DIR / "data_inventory.csv",
    "fred_annualized": INTERIM_DIR / "fred_annualized.csv",
    "national_housing_income": PROCESSED_DIR / "national_housing_income.csv",
    "commute_fred_clean": INTERIM_DIR / "commute_fred_clean.csv",
    "commute_census_clean": INTERIM_DIR / "commute_census_clean.csv",
    "national_story_series": PROCESSED_DIR / "national_story_series.csv",
    "commute_story_series": PROCESSED_DIR / "commute_story_series.csv",
    "housing_access_geo_snapshot": PROCESSED_DIR / "housing_access_geo_snapshot.csv",
}

loaded = {name: pd.read_csv(path) for name, path in files.items()}
file_summary = pd.DataFrame([
    {"dataset": name, "path": str(path.relative_to(ROOT)), "rows": len(loaded[name]), "columns": len(loaded[name].columns)}
    for name, path in files.items()
])
file_summary

## B. National Housing/Income Series Review

The national file has annualized values. Monthly, weekly, and quarterly FRED series were converted to annual averages. Raw frequencies remain preserved in `data/raw/fred/`.

Key validation points:

- Nominal and real housing values are labeled separately.
- CPI adjustment is present through `cpi_index` and `cpi_base_year`.
- `MEHOINUSA672N` is already real median household income and was not inflation-adjusted again.
- `home_price_to_real_income_ratio` is available and cleanly computed.
- Mortgage rates are annual averages; 2026 is partial and should not be used as a final comparison endpoint.
- Use 2024 as the clean common endpoint for national story comparisons because income ends in 2024 and CPI base year is 2024.

In [ ]:
national = loaded["national_story_series"]
important_national = [
    "median_home_price_nominal",
    "real_median_home_price",
    "real_median_household_income",
    "home_price_to_real_income_ratio",
    "mortgage_rate_annual_avg",
    "case_shiller_index",
    "national_mean_commute_minutes",
]

coverage = []
for col in important_national:
    s = national[["year", col]].dropna()
    coverage.append({
        "variable": col,
        "first_year": int(s["year"].min()) if not s.empty else None,
        "last_year": int(s["year"].max()) if not s.empty else None,
        "non_null_years": len(s),
        "missing_years": int(national[col].isna().sum()),
    })
pd.DataFrame(coverage)

In [ ]:
national[[
    "year", "cpi_index", "cpi_base_year", "median_home_price_nominal",
    "real_median_home_price", "real_median_household_income",
    "home_price_to_real_income_ratio", "mortgage_rate_annual_avg",
    "case_shiller_index",
]].tail(8)

### Diagnostic National Charts

These charts are diagnostic only and are saved in `reports/eda_review_charts/`.

![Real median home price](../reports/eda_review_charts/diagnostic_real_median_home_price.png)

![Real median household income](../reports/eda_review_charts/diagnostic_real_median_household_income.png)

![Home price to income ratio](../reports/eda_review_charts/diagnostic_home_price_income_ratio.png)

![Mortgage rate](../reports/eda_review_charts/diagnostic_mortgage_rate.png)

![Case-Shiller index](../reports/eda_review_charts/diagnostic_case_shiller_index.png)

## C. Commute/Access Series Review

FRED commute observations were not sufficient as the primary commute source in this run. The usable commute data comes from controlled Census ACS API pulls.

Key validation points:

- National ACS commute trend exists for 2005-2019 and 2021-2024.
- 2020 is omitted.
- The denominator `B08012_001E` is nonzero for loaded observations.
- Computed national mean commute values are plausible, ranging from roughly 25 to 28 minutes.
- COVID and post-COVID years need annotation because remote work changed the commuting universe.
- Commute data is usable both as a recent national trend and as a 2023 selected-geography snapshot, but not as a 1960-present trend.

In [ ]:
commute = loaded["commute_story_series"]
commute_summary = commute.groupby(["geo_level", "geography"], as_index=False).agg(
    first_year=("year", "min"),
    last_year=("year", "max"),
    rows=("year", "count"),
    min_commute=("mean_commute_minutes", "min"),
    max_commute=("mean_commute_minutes", "max"),
    zero_denominators=("B08012_001E", lambda s: int((s == 0).sum())),
    missing_commute=("mean_commute_minutes", lambda s: int(s.isna().sum())),
)
commute_summary

In [ ]:
commute[commute["geo_level"].eq("national")][["year", "B08012_001E", "mean_commute_minutes"]]

### Diagnostic Commute Charts

![National mean commute time](../reports/eda_review_charts/diagnostic_national_commute_time.png)

![Selected geography commute comparison](../reports/eda_review_charts/diagnostic_selected_geo_commute_comparison.png)

![Ranked commute burden](../reports/eda_review_charts/diagnostic_ranked_commute_burden.png)

## D. Geographic Snapshot Review

The housing + access snapshot uses selected county proxies, not full metro areas. Each selected geography has both housing and commute data for the 2023 ACS 5-year release.

Available housing affordability variables:

- `median_home_value`
- `median_household_income`
- `home_value_to_income_ratio`

Available commute/access variable:

- `mean_commute_minutes`

A scatter/quadrant visual is feasible using `home_value_to_income_ratio` on the x-axis and `mean_commute_minutes` on the y-axis. The final chart should label the geography type clearly as selected county proxies.

In [ ]:
geo = loaded["housing_access_geo_snapshot"]
geo[[
    "year", "case_geography", "NAME", "geo_level", "mean_commute_minutes",
    "median_home_value", "median_household_income", "home_value_to_income_ratio",
    "housing_access_quadrant",
]].sort_values("mean_commute_minutes", ascending=False)

### Diagnostic Housing + Access Scatter

Median reference lines are useful here because they create an interpretable quadrant without inventing a black-box score.

![Housing access scatter](../reports/eda_review_charts/diagnostic_housing_access_scatter.png)

## E. Candidate Derived Metrics

Recommended national metrics:

- `real_median_home_price`: main long-run sticker-price pressure measure.
- `real_median_household_income`: ability-to-pay context, already real.
- `home_price_to_real_income_ratio`: clearest national affordability-pressure ratio.
- `mortgage_rate_annual_avg`: financing context, useful but not the central claim.
- `case_shiller_index`: optional repeat-sales cross-check.

Recommended commute/access metrics:

- `mean_commute_minutes`: main access-burden measure.
- selected-geography commute rank: useful for a simple comparison.
- commute burden relative to selected-geography median: optional annotation only.

Recommended combined lens:

Use `home_value_to_income_ratio` and `mean_commute_minutes` directly in a quadrant/scatter chart. Do not use a black-box composite score.

In [ ]:
metric_check = pd.DataFrame([
    {"metric": "real_median_home_price", "recommended": True, "role": "long-run sticker-price pressure"},
    {"metric": "real_median_household_income", "recommended": True, "role": "ability-to-pay context"},
    {"metric": "home_price_to_real_income_ratio", "recommended": True, "role": "national affordability-pressure ratio"},
    {"metric": "mortgage_rate_annual_avg", "recommended": True, "role": "financing context"},
    {"metric": "case_shiller_index", "recommended": False, "role": "optional cross-check"},
    {"metric": "mean_commute_minutes", "recommended": True, "role": "access burden"},
    {"metric": "home_value_to_income_ratio", "recommended": True, "role": "geographic housing pressure"},
    {"metric": "housing_access_quadrant", "recommended": True, "role": "interpretable combined lens"},
])
metric_check

## F. Candidate Visual Storylines

The strongest story structure is sequential:

1. Establish that the sticker price has risen in real terms.
2. Add income context to show why sticker price alone is incomplete.
3. Add mortgage context only where it clarifies interpretation.
4. Introduce commute/access as a recent burden lens.
5. Show selected geography variation in commute burden.
6. End with the housing + access quadrant as the main synthesis.

This creates a guided article instead of a dashboard.

## G. Recommended Final Visuals

Recommended final visuals for the Streamlit data story:

1. **The Sticker Price Story**
   - Data: `data/processed/national_story_series.csv`
   - Variables: `year`, `real_median_home_price`
   - Type: line chart
   - Message: real home prices show long-run upward pressure.
   - Status: essential

2. **Home Prices Outrunning Paychecks**
   - Data: `data/processed/national_story_series.csv`
   - Variables: `year`, `home_price_to_real_income_ratio`
   - Type: line chart
   - Message: income context makes the affordability pressure more interpretable.
   - Status: essential

3. **Mortgage Rates as Context**
   - Data: `data/processed/national_story_series.csv`
   - Variables: `year`, `mortgage_rate_annual_avg`
   - Type: compact line chart or annotated context chart
   - Message: financing conditions affect interpretation.
   - Status: optional but useful

4. **Commute Time in the ACS Era**
   - Data: `data/processed/commute_story_series.csv`
   - Variables: `year`, `mean_commute_minutes`, national filter
   - Type: line chart
   - Message: access burden can be shown as a recent trend.
   - Status: essential

5. **Selected Geography Commute Burden**
   - Data: `data/processed/housing_access_geo_snapshot.csv`
   - Variables: `case_geography`, `mean_commute_minutes`
   - Type: ranked bar chart
   - Message: access burden varies across selected places.
   - Status: essential if geographic lens is approved

6. **Housing + Access Quadrant**
   - Data: `data/processed/housing_access_geo_snapshot.csv`
   - Variables: `home_value_to_income_ratio`, `mean_commute_minutes`, `case_geography`, `housing_access_quadrant`
   - Type: scatterplot with median reference lines
   - Message: combined burden is more revealing than sticker price alone.
   - Status: essential centerpiece

7. **Case-Shiller Cross-Check**
   - Data: `data/processed/national_story_series.csv`
   - Variables: `year`, `case_shiller_index`
   - Type: supporting line chart
   - Message: repeat-sales index reinforces the broad housing-pressure pattern.
   - Status: optional

## H. Risks and Limitations

- Commute/access data does not cover 1960-present.
- The national commute trend starts in 2005 and omits 2020.
- Post-2020 commute values require remote-work context.
- Selected geographies are county proxies, not full metro areas.
- National FRED series have different endpoints; 2024 is the recommended clean comparison endpoint.
- Real median household income is already inflation-adjusted and should not be deflated again.
- The final story should avoid causal language and use careful phrasing such as “suggests,” “adds context,” and “reveals a tradeoff.”

## I. Final Proceed Decision

**Proceed with caveats.**

The data supports the project if the final story is framed as long-run housing affordability pressure plus a recent commute/access lens. The recommended centerpiece is the housing + access quadrant using selected county proxies, provided the proxy geography limitation is acceptable.

The next phase can move into visual prototyping and then Streamlit implementation after user approval of:

- county proxies versus metro-level matching,
- whether national charts should cap at 2024,
- whether Case-Shiller should appear in the final app,
- whether optional shelter and transportation CPI should remain outside the main story.